In [ ]:
'''
python version 3.10.12
'''

In [7]:
'''
Make sure to confirm the full path to the requirements.txt file. 
'''
! pip install -r requirements.txt
'''
Please restart this file after executing this cell !!!
'''

In [1]:
import argparse
import numpy as np
import os
from pathlib import Path
import torch
import torch.backends.cudnn as cudnn
from models.model import Network
from func.functions import evaluate
from ASVDataloader.ASVRawTest import ASVRawTest
from utils.utils import Genotype
import time

In [4]:
def get_score(eval_path,data_base,comment):
    OUTPUT_CLASSES = 2
    model_path = 'change this path to your RawPCDARTS model' # download here [https://huggingface.co/VoiceWukong/VoiceWukong/resolve/main/RawPCDARTS.pth?download=true]
    checkpoint = torch.load(model_path)
    genotype = eval("Genotype(normal=[('sep_conv_5x5', 0), ('sep_conv_5x5', 1), ('sep_conv_3x3', 2), ('dil_conv_5x5', 0), ('avg_pool_3x3', 2), ('max_pool_3x3', 3), ('avg_pool_3x3', 2), ('avg_pool_3x3', 4)], normal_concat=range(2, 6), reduce=[('sep_conv_5x5', 1), ('sep_conv_3x3', 0), ('dil_conv_3x3', 2), ('avg_pool_3x3', 0), ('dil_conv_5x5', 2), ('sep_conv_3x3', 3), ('dil_conv_3x3', 2), ('avg_pool_3x3', 4)], reduce_concat=range(2, 6))")
    front_end='LFCC'
    init_channels=16
    layers=4
    nftt=1024
    hop=4
    nfilter=70
    num_ceps=20
    is_log=True
    is_cmvn=False
    sr=16000
    model = Network(init_channels, layers, nftt,hop,sr,is_log,is_cmvn,nfilter,num_ceps, OUTPUT_CLASSES, genotype, front_end)
    model.drop_path_prob = 0.0
    eval_protocol=eval_path
    data=data_base
    eval_dataset=ASVRawTest(Path(data), 'eval', eval_protocol)
    model = model.cuda()
    model.load_state_dict(checkpoint)
    eval_loader = torch.utils.data.DataLoader(
        dataset=eval_dataset,
        batch_size=24,
        num_workers=0,
        pin_memory=True,
        shuffle=False,
        drop_last=False,
    )
    evaluate(eval_loader, model,comment)
    

In [ ]:
'''
    |- path to VoiceWukong dataset
        |- Alldataset
        |- Alldataset32K
        |- ...
'''
data_path='change this to your VoiceWukong dataset'
get_score('change this to your eval_list.txt path',data_path,'en')
get_score('change this to your zh_eval_list.txt path',data_path,'zh')


'''
en_eval_socre.txt and zh_eval_score.txt will be generated in the current directory
'''
